# 01 — Historical archive

The competition exposes a historical archive so you can experiment **offline**:
one gzip-JSONL file per `event_type` × `quarter`, each line a past event with its
disclosure. This notebook:

1. Browses the archive manifest (`GET /archive`)
2. Downloads and **caches** the files locally (`data/archive/`, gitignored)
3. Loads them into a pandas DataFrame
4. Inspects a single event and its disclosure

With a live key you pull the real archive; without one, you load the small bundled
sample so the mechanics are still clear.

> Download URLs are short-lived and signed. The helper refreshes an expired URL
> automatically (via `GET /archive/{event_type}/{quarter}`) when you pass it a
> client.

In [ ]:
from pathlib import Path

from IPython.display import display

from examples import Client, load_config
from examples.archive import download_archive, load_archive
from examples.frames import manifest_frame

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sample").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache
SAMPLE_DIR = REPO / "data" / "sample"  # tiny bundled stand-in

config = load_config()
print("Mode:", "LIVE" if config.is_live else "SAMPLE (no EM_API_KEY — using bundled data)")

## 1. Browse the manifest

Each row is one downloadable file — what event type and quarter it covers, how
many events it holds, and how big it is. (The signed URLs are omitted from this
view.)

In [ ]:
manifest = None
if config.is_live:
    with Client.from_env() as client:
        manifest = client.archive_manifest()
    display(manifest_frame(manifest))
else:
    print("Sample mode: no live manifest. We'll load the bundled sample archive below.")

## 2. Download & cache

`download_archive` writes each file into `data/archive/`, **skipping** any file
already cached at the size the manifest reports — so re-running is cheap. Narrow
the pull with `only=`, e.g. `only=lambda f: f.quarter >= "2026Q1"`.

In [ ]:
if config.is_live:
    with Client.from_env() as client:
        paths = download_archive(
            manifest,
            ARCHIVE_DIR,
            client=client,  # lets it refresh expired URLs
            # only=lambda f: f.event_type == "EARNINGS_RELEASE",
        )
    source = ARCHIVE_DIR
    print(f"{len(paths)} file(s) cached in {ARCHIVE_DIR}")
else:
    source = SAMPLE_DIR
    print(f"Sample mode: reading the bundled archive in {SAMPLE_DIR}")

## 3. Load into pandas

`load_archive` reads every `*.jsonl.gz` under a directory (or a single file) into
one DataFrame, adding `event_type` and `quarter` provenance columns from each
filename.

In [ ]:
df = load_archive(source)
print(f"Loaded {len(df):,} events")
print("Columns:", list(df.columns))
df.head()

## 4. Inspect one event

Pull a single row and read its focal assets and disclosure facts. (The bundled
sample inlines disclosures under a `disclosure` field; the live archive may carry
additional fields — inspect `df.columns` to see what a given dataset provides.)

In [ ]:
row = df.iloc[0]

focal = row.get("focal_assets") if "focal_assets" in df.columns else None
tickers = [a["identifier_value"] for a in (focal or [])]
print(f"event_type={row['event_type']}  quarter={row['quarter']}  tickers={tickers}")
print(f"event_datetime={row.get('event_datetime')}\n")

disclosure = row.get("disclosure") if "disclosure" in df.columns else None
if isinstance(disclosure, dict):
    for item in disclosure.get("items", []):
        if item.get("kind") == "facts":
            print(f"facts from {item.get('source')!r}:")
            for fact in item["content"]:
                print("  -", fact)
else:
    print("No inlined 'disclosure' column here — check df.columns for this dataset's fields.")

## 5. Reproduce the baseline scoring regressions

`examples.scoring` ports the competition's **exact** scoring transform, so you can
(i) reproduce the leaderboard's numbers offline and (ii) use it in your own
analyses and optimizations. Scoring runs *within a period* (a quarter): the
realized one-day cumulative abnormal return (`car1`) of every outcome is
percentile-ranked into the dependent variable `y = r_abn`, and the earnings
surprise is percentile-ranked into the `s` regressor. Each reference baseline
already submitted a predicted percentile `ĝ`.

We reproduce **Table 3** of Koijen & Levy (WP) on the disseminated data: `y`
regressed on five specifications —

1. surprise only (`s`) — the naive benchmark,
2. Gemini only (`ĝ`),
3. GPT-5 nano only (`ĝ`),
4. surprise + Gemini,
5. surprise + GPT-5 nano.

This section needs the **live** archive (the bundled sample carries no returns or
baseline predictions). We use `2025Q4`, the sealed quarter behind the paper's
table. (The paper also reports bootstrapped R² intervals; we show point estimates
and standard errors.)

In [ ]:
from examples.scoring import BASELINE_LABELS, add_percentiles, outcomes_frame

PERIOD = "2026Q3"

needed = {"event_returns", "metrics", "baseline_predictions"}
have_scoring_data = needed <= set(df.columns) and (df["quarter"] == PERIOD).any()

if have_scoring_data:
    # Percentiles are ranked WITHIN the period, so restrict to the quarter first.
    records = df[df["quarter"] == PERIOD].to_dict("records")
    scored = add_percentiles(outcomes_frame(records))
    models = list(BASELINE_LABELS.values())  # ["Gemini 2.5 Flash-Lite", "GPT-5 nano"]
    # The paper's single sample: outcomes with y, a surprise, and both baselines.
    fit_df = scored.dropna(subset=["y", "surprise_pct", *models]).copy()
    print(f"{PERIOD}: {len(scored):,} scored outcomes | fit sample N = {len(fit_df):,}")
else:
    print(f"Sample mode: section 5 needs the live archive for {PERIOD}. Skipping.")

In [ ]:
if have_scoring_data:
    import pandas as pd
    import statsmodels.api as sm
    from stargazer.stargazer import Stargazer

    gemini, gpt = models  # display names, in BASELINE_LABELS order
    S, G = "s (surprise pctile)", "ĝ (model pctile)"

    def fit(cols):
        # cols: {display_name -> source column}; shared names let stargazer stack rows.
        X = sm.add_constant(pd.DataFrame({name: fit_df[src] for name, src in cols.items()}))
        return sm.OLS(fit_df["y"].to_numpy(), X).fit()

    fits = [
        fit({S: "surprise_pct"}),  # (1) Baseline: surprise only
        fit({G: gemini}),  # (2) Gemini only
        fit({G: gpt}),  # (3) GPT-5 nano only
        fit({S: "surprise_pct", G: gemini}),  # (4) surprise + Gemini
        fit({S: "surprise_pct", G: gpt}),  # (5) surprise + GPT-5 nano
    ]

    star = Stargazer(fits)
    star.title(f"Transcript summaries — Koijen & Levy (WP) Table 3, reproduced on {PERIOD}")
    star.custom_columns(["Baseline", gemini, gpt, gemini, gpt], [1, 1, 1, 1, 1])
    star.covariate_order(["const", S, G])
    star.rename_covariates({"const": "Intercept"})
    star.dependent_variable_name("r_abn  (percentiled CAR1)")
    star.show_model_numbers(True)
    display(star)

## 6. Backtest your own `predict.py`

Runs your actual prediction strategy — the same `_ask_llm` your deployed handler
calls — against the archive instead of a live webhook payload. The archive has
no `information_url`; its facts live inline under `disclosure.items[].content`,
so we rebuild an equivalent summary text from there.

Results are cached to disk (`backtest_cache.json`), keyed by `event_id`, so a
kernel restart or a widened sample doesn't re-pay for events you already scored.
Start with a small sample (`N_SAMPLE` below) before running a full quarter —
each call is a real, billed Kimi K3 request.

In [ ]:
import sys
import time

period_df = df[df["quarter"] == PERIOD].copy()  # PERIOD/df/scored come from Section 5

REPO_ROOT = REPO.parent  # starter-modal/ — one level above examples/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import predict  # noqa: E402  (your actual predict.py — same code the deployed handler runs)

def facts_to_summary(disclosure: dict) -> str:
    """Rebuild an information_url-shaped summary string from an archive disclosure."""
    lines = []
    for item in disclosure.get("items", []):
        if item.get("kind") == "facts":
            lines.extend(item["content"])
    return "\n".join(f"- {f}" for f in lines)

print("KIMI_API_KEY set:", bool(__import__("os").environ.get("KIMI_API_KEY")))

### Peer momentum helper

For each event, the mean next-day abnormal return (`car1`) of same-sector peers
that already reported *earlier in the same quarter* — a mild sector-drift signal
to hand the LLM alongside the event's own facts.

Sector is resolved via yfinance, disk-cached to `data_provider/sector_cache.json`
(sectors don't change day to day, so a rerun or future session never re-fetches
a ticker once seeded). Prefetched **sequentially** here, before the threaded
backtest loop below — concurrent `.info` calls from multiple worker threads at
once is what rate-limited yfinance last time, not the ticker count itself.

In [ ]:
from data_provider.yfinance import get_sector

unique_tickers = sorted({
    a["identifier_value"]
    for _, r in period_df.iterrows()
    for a in r["focal_assets"]
})
print(f"Prefetching sector for {len(unique_tickers)} unique tickers (sequential, disk-cached)...")
for i, t in enumerate(unique_tickers, 1):
    get_sector(t)
    if i % 25 == 0 or i == len(unique_tickers):
        print(f"  {i}/{len(unique_tickers)} sectors resolved")

def peer_momentum(row, period_df) -> float | None:
    """Mean car1 of same-sector events that reported strictly earlier in the quarter."""
    ticker = row["focal_assets"][0]["identifier_value"]
    sector = get_sector(ticker)
    if not sector:
        return None

    prior = period_df[period_df["event_datetime"] < row["event_datetime"]]
    car1s = []
    for _, prow in prior.iterrows():
        p_ticker = prow["focal_assets"][0]["identifier_value"]
        if get_sector(p_ticker) != sector:
            continue
        leg = (prow.get("event_returns") or {}).get(p_ticker) or {}
        if leg.get("car1") is not None:
            car1s.append(leg["car1"])

    return sum(car1s) / len(car1s) if car1s else None

### Surprise-rank helper (EXPERIMENTAL, backtest-only)

Quarter-to-date percentile rank of this event's EPS/revenue surprise
(`metrics.earnings_surprise.surprise`) against every event that reported
*strictly earlier* in the same quarter -- same no-lookahead pattern as
peer_momentum above.

This is only possible in the backtest: the archive carries the surprise
number, but the live webhook event does not, and there's no consensus-estimate
source wired up yet (FMP's analyst-estimates endpoint is paywalled on the free
tier). So `predict.py`'s `surprise_signal` param defaults to None in the live
path -- this cell exists purely to measure whether the signal is strong enough
to justify finding a live consensus source at all. (Standalone analysis: see
`explore_text_alpha.py` -- `surprise_pct`, percentile-ranked within a full
sealed quarter, had R²=6.3% against realized car1, the single strongest
feature found, vs ~1% for text-keyword sentiment.)

In [ ]:
def surprise_rank(row, period_df) -> float | None:
    """Quarter-to-date percentile rank of this event's surprise vs strictly-earlier events."""
    own = (row.get("metrics") or {}).get("earnings_surprise", {}).get("surprise")
    if own is None:
        return None

    prior = period_df[period_df["event_datetime"] < row["event_datetime"]]
    prior_surprises = [
        s for _, prow in prior.iterrows()
        if (s := (prow.get("metrics") or {}).get("earnings_surprise", {}).get("surprise")) is not None
    ]
    if not prior_surprises:
        return None

    return sum(1 for s in prior_surprises if s < own) / len(prior_surprises)

In [ ]:
import json
import os
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

CACHE_PATH = Path.cwd() / "backtest_cache_surprise.json"  # new file again -- adding surprise_signal
# changes what gets sent to the LLM, so mixing with older cached (pre-surprise-signal)
# predictions under the same event_id would silently corrupt the A/B comparison.
MAX_WORKERS = 10  # concurrent LLM calls -- tune down if you hit rate limits

def load_cache(path: Path) -> dict:
    if not path.exists() or path.stat().st_size == 0:
        return {}
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        print(f"[WARN] {path.name} was empty/corrupt (likely an interrupted write) -- starting fresh.")
        return {}

def save_cache(path: Path, data: dict) -> None:
    # Write to a temp file and atomically replace -- a kernel interrupt mid-write
    # can no longer truncate the real cache file.
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2))
    os.replace(tmp, path)

cache = load_cache(CACHE_PATH)
cache_lock = threading.Lock()

N_SAMPLE = 100  # near-full quarter (772 events) -- keeps n_obs stable/comparable run to run,
# instead of bouncing around on a small, randomly-varying covered subset
sample_df = period_df.sample(n=min(N_SAMPLE, len(period_df)), random_state=0)

todo = [row for _, row in sample_df.iterrows() if row["event_id"] not in cache]
print(f"{len(sample_df) - len(todo)} already cached, {len(todo)} to run now")

def predict_one(row):
    summary_text = facts_to_summary(row["disclosure"])
    tickers = [a["identifier_value"] for a in row["focal_assets"]]

    momentum = peer_momentum(row, period_df)
    momentum_text = (
        f"{momentum:.4f}" if momentum is not None
        else "no same-sector peers reported yet this quarter"
    )

    surprise = surprise_rank(row, period_df)
    surprise_text = (
        f"{surprise:.2f}" if surprise is not None
        else "no surprise data / no prior events to rank against"
    )

    preds = {}
    for ticker in tickers:
        print(f"[DEBUG] event={row['event_id']} ticker={ticker} as_of={row['knowledge_cutoff']!r} peer_momentum={momentum_text} surprise_rank={surprise_text}")
        preds[ticker] = predict._ask_llm(
            summary={"summary": summary_text},
            ticker=ticker,
            event_type=row["event_type"],
            as_of=row["knowledge_cutoff"],
            peer_context=momentum_text,
            surprise_signal=surprise_text,
        )
    return row["event_id"], preds

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(predict_one, row): row["event_id"] for row in todo}
    done = 0
    failed = []
    for future in as_completed(futures):
        event_id = futures[future]  # known even if the call below raises
        try:
            event_id, preds = future.result()
        except Exception as e:
            # A single timeout/API error used to crash the whole cell here --
            # every other future still running in the background would finish
            # (the executor waits for them on exit) but never get cached, since
            # the loop had already exited. Now we just log and move on: this
            # event_id stays out of the cache, so it's automatically retried
            # the next time you run this cell (it'll show up in 'todo' again).
            print(f"[ERROR] event={event_id} failed: {e!r} -- skipping, not cached")
            failed.append(event_id)
            continue
        with cache_lock:
            cache[event_id] = preds
            save_cache(CACHE_PATH, cache)  # still resume-safe if interrupted mid-run
        done += 1
        if done % 10 == 0 or done == len(todo):
            print(f"{done}/{len(todo)} done")

print(f"{len(cache)} events predicted (cached in {CACHE_PATH.name})")
if failed:
    print(f"{len(failed)} event(s) failed and were skipped: {failed}")
    print("Re-run this cell to retry them -- they were never added to the cache.")


In [ ]:
rows = [
    {"event_id": event_id, "identifier_value": ticker, "predicted": pred}
    for event_id, preds in cache.items()
    for ticker, pred in preds.items()
]
pred_df = pd.DataFrame(rows)

# Join on (event_id, identifier_value) — `scored` has one row per (event, asset),
# so joining on event_id alone would mis-match events with multiple tickers.
merged = scored.merge(pred_df, on=["event_id", "identifier_value"], how="left")

covered = merged["predicted"].notna().sum()
print(f"{covered} / {len(pred_df)} cached predictions matched to a scored outcome\n")

# examples.scoring.score_submission is the exact port of the contest's own
# scorer (Official Rules §5) — same metric family as the leaderboard, and
# directly comparable to the Gemini/GPT-5-nano numbers from Section 5.
from examples.scoring import score_submission

result = score_submission(merged, "predicted")
for key in ("n_obs",  "r_squared", "r_squared_surprise", "delta_r_squared"):
    print(f"{key:>20}: {result[key]}")
print()
for key in ("r_squared_imputed", "delta_r_squared_imputed", "imputed_event_count"):
    print(f"{key:>20}: {result[key]}")


## 7. Compare against Table 3 baselines

Same PERIOD, same `y`. Baseline numbers are hardcoded from Section 5's Table 3
reproduction (N=1849, full quarter) — your model's bars are your current
`result` from Section 6 and are only sample-size-comparable once `n_obs`
approaches 1849.

In [ ]:
import matplotlib.pyplot as plt

# Reference numbers from Koijen & Levy (WP) Table 3 — N=1849, full quarter
labels = [
    "Surprise only\n(N=1849)",
    "Gemini\n(N=1849)",
    "GPT-5 nano\n(N=1849)",
    f"Your model\n(N={result['n_obs']}, ungefuellt)",
]
r2_values = [0.049, 0.098, 0.072, result["r_squared"]]
delta_values = [0.0, 0.098 - 0.049, 0.072 - 0.049, result["delta_r_squared"]]
colors = ["#888888", "#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(labels, r2_values, color=colors)
axes[0].set_ylabel("R^2 (surprise + model, two-regressor)")
axes[0].set_title("R^2 vs. Table 3 baselines")
axes[0].tick_params(axis="x", labelrotation=15)

axes[1].bar(labels, delta_values, color=colors)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Delta R^2 over surprise-only")
axes[1].set_title("Incremental alpha over surprise")
axes[1].tick_params(axis="x", labelrotation=15)

plt.tight_layout()
plt.show()

print(
    f"Baselines: N=1849 (full quarter). Your model: N={result['n_obs']} (ungefuellt) "
    "-- only directionally comparable until n_obs approaches 1849.\n"
    f"Official contest metric (imputed, full N=1849, {result['imputed_event_count']} mean-filled): "
    f"delta_r_squared_imputed = {result['delta_r_squared_imputed']:.5f}"
)

## 8. Log this run (tracks ranking-relevant metric over time)

The contest ranks on `delta_r_squared` over the **common sample** (Official
Rules §5 / imputed family) — every run here gets appended to
`delta_r2_log.csv` so you can see whether changes to `predict.py` (prompt,
reasoning effort, added data sources) are actually moving that number, instead
of eyeballing one run at a time.

In [ ]:
import csv
from datetime import datetime, timezone

LOG_PATH = Path.cwd() / "delta_r2_log.csv"
RUN_NOTES = ""  # fill in before running, e.g. "added yfinance trend+vol, reasoning=high"

row = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "period": PERIOD,
    "n_obs": result["n_obs"],
    "r_squared": result["r_squared"],
    "r_squared_surprise": result["r_squared_surprise"],
    "delta_r_squared": result["delta_r_squared"],
    "r_squared_imputed": result["r_squared_imputed"],
    "delta_r_squared_imputed": result["delta_r_squared_imputed"],
    "imputed_event_count": result["imputed_event_count"],
    "notes": RUN_NOTES,
}

is_new = not LOG_PATH.exists()
with LOG_PATH.open("a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(row.keys()))
    if is_new:
        writer.writeheader()
    writer.writerow(row)

history = pd.read_csv(LOG_PATH)
print(f"Logged run #{len(history)} to {LOG_PATH.name}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.index, history["delta_r_squared"], marker="o", label="delta_r_squared (ungefuellt)")
ax.plot(history.index, history["delta_r_squared_imputed"], marker="o", label="delta_r_squared_imputed (contest metric)")
ax.axhline(0.049, color="gray", linestyle="--", linewidth=1, label="Gemini baseline (0.049)")
ax.set_xlabel("run #")
ax.set_ylabel("delta R^2")
ax.set_title("Delta R^2 over time")
ax.legend()
plt.tight_layout()
plt.show()

history.tail(10)

## Where to go from here

The archive gives you the **inputs** — events and their disclosures. To turn this
into a backtest you'd:

1. Pair each event with the realized next-day reaction for its focal asset, from
   your own market-data source.
2. Convert that reaction into a percentile rank in `[0, 1]` **across the
   quarter's cross-section** of scored event outcomes — a rank against all the
   period's events, not against the asset's own history. That's the target the
   competition scores; `examples.scoring.percentile_ranks` is the exact
   transform (see section 5).
3. Score your model's predicted percentile against it offline, and iterate.

When your model is ready, move it into a deployed webhook handler (see the starter
repos) — that's what receives live events and submits predictions. These notebooks
deliberately stop short of `POST /predictions`.